## 10_v4_inputs
  Prepares the HU and SK settlement-level inputs: LAU unit table (population, area, centroid)
  and the total number of commuters N_c per country.
  Output: data/interim/10_v4_inputs/, data/processed/10_v4_inputs/nc_summary.csv.

In [ ]:
import json
import re
import pandas as pd
from pathlib import Path

# Config: change only here
PROJECT_ROOT = Path("..")          # notebooks run from the notebooks folder
LAU_FILE = "data/raw/gisco/LAU_RG_01M_2021_3035.geojson"
OUT_DIR, PROPS_FILE = "data/interim/10_v4_inputs", "hu_sk_lau_gisco_props.parquet"
COUNTRIES = ["HU", "SK"]
GISCO_COLS = ["CNTR_CODE", "LAU_ID", "LAU_NAME", "POP_2021"]
COUNTRY_COL, ID_COL, POP_COL = "CNTR_CODE", "LAU_ID", "POP_2021"

out_dir = PROJECT_ROOT / OUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)
props_path = out_dir / PROPS_FILE
props_rel = (Path(OUT_DIR) / PROPS_FILE).as_posix()

if props_path.is_file():
    props = pd.read_parquet(props_path)
    print("Loaded from file:", props_rel)
elif not (PROJECT_ROOT / LAU_FILE).is_file():
    props = None
    print("Raw file missing:", LAU_FILE)
    print("Manual fix: put the GISCO LAU 2021 GeoJSON (EPSG:3035) at that path, then rerun the cell.")
else:
    # Parse only the 'properties' blocks of the big file (no geometry), keep HU and SK
    text = (PROJECT_ROOT / LAU_FILE).read_text(encoding="utf-8")
    recs = []
    for hit in re.finditer(r'"properties"\s*:\s*\{[^{}]*\}', text):
        block = json.loads(hit.group(0)[hit.group(0).index("{"):])
        if block.get(COUNTRY_COL) in COUNTRIES:
            recs.append({c: block.get(c) for c in GISCO_COLS})
    del text
    props = pd.DataFrame(recs)
    props.to_parquet(props_path, index=False)
    print("Parsed and saved:", props_rel)

if props is not None:
    props[ID_COL] = props[ID_COL].astype(str)
    props[POP_COL] = pd.to_numeric(props[POP_COL], errors="coerce")
    print("Units per country:", props[COUNTRY_COL].value_counts().reindex(COUNTRIES).to_dict())
    print("Missing values:", props[GISCO_COLS].isna().sum().to_dict())
    print("Duplicated (country, id) pairs:", int(props.duplicated([COUNTRY_COL, ID_COL]).sum()))
    print("Population sum per country:", props.groupby(COUNTRY_COL)[POP_COL].sum().astype(int).to_dict())

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from shapely.geometry import shape

# Config: change only here
PROJECT_ROOT = Path("..")          # notebooks run from the notebooks folder
LAU_FILE = "data/raw/gisco/LAU_RG_01M_2021_3035.geojson"
OUT_DIR, CENT_FILE = "data/interim/10_v4_inputs", "hu_sk_lau_centroids.parquet"
COUNTRIES = ["HU", "SK"]
COUNTRY_COL, ID_COL, AREA_COL, POP_COL = "CNTR_CODE", "LAU_ID", "AREA_KM2", "POP_2021"
PERCENTILES = [0.01, 0.5, 0.99]

out_dir = PROJECT_ROOT / OUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)
cent_path = out_dir / CENT_FILE
cent_rel = (Path(OUT_DIR) / CENT_FILE).as_posix()

if cent_path.is_file():
    cent = pd.read_parquet(cent_path)
    print("Loaded from file:", cent_rel)
elif not (PROJECT_ROOT / LAU_FILE).is_file():
    cent = None
    print("Raw file missing:", LAU_FILE)
    print("Manual fix: put the GISCO LAU 2021 GeoJSON (EPSG:3035) at that path, then rerun the cell.")
else:
    # Stream the features one by one, build geometry only for HU and SK
    text = (PROJECT_ROOT / LAU_FILE).read_text(encoding="utf-8")
    decoder, n_text = json.JSONDecoder(), len(text)
    pos = text.index("[", text.index('"features"')) + 1
    recs, n_all = [], 0
    while True:
        while pos < n_text and text[pos] in " ,\n\r\t":
            pos += 1
        if pos >= n_text or text[pos] == "]":
            break
        feat, pos = decoder.raw_decode(text, pos)
        n_all += 1
        pr = feat["properties"]
        if pr.get(COUNTRY_COL) not in COUNTRIES:
            continue
        geom = shape(feat["geometry"])
        c = geom.centroid
        recs.append({ID_COL: str(pr[ID_COL]), COUNTRY_COL: pr[COUNTRY_COL], POP_COL: pr.get(POP_COL),
                     AREA_COL: pr.get(AREA_COL), "x_3035": c.x, "y_3035": c.y,
                     "area_geom_km2": geom.area / 1e6, "geom_valid": bool(geom.is_valid)})
    del text
    cent = pd.DataFrame(recs)
    cent.to_parquet(cent_path, index=False)
    print(f"Parsed {n_all} features, kept {len(cent)} HU/SK units; saved:", cent_rel)

if cent is not None:
    cent[ID_COL] = cent[ID_COL].astype(str)
    print("Units per country:", cent[COUNTRY_COL].value_counts().reindex(COUNTRIES).to_dict())
    print("Invalid geometries:", int((~cent["geom_valid"]).sum()),
          "| missing centroid:", int(cent[["x_3035", "y_3035"]].isna().any(axis=1).sum()))
    ratio = (cent["area_geom_km2"] / pd.to_numeric(cent[AREA_COL], errors="coerce")).replace([np.inf, -np.inf], np.nan)
    print("Geometric area / AREA_KM2 per country:")
    print(ratio.groupby(cent[COUNTRY_COL]).quantile(PERCENTILES).unstack().reindex(COUNTRIES).round(3).to_string())

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Config: change only here
PROJECT_ROOT = Path("..")          # notebooks run from the notebooks folder
OUT_DIR = "data/interim/10_v4_inputs"
PROPS_FILE, CENT_FILE, UNITS_FILE = "hu_sk_lau_gisco_props.parquet", "hu_sk_lau_centroids.parquet", "hu_sk_lau_units.parquet"
COUNTRY_COL, ID_COL, NAME_COL, POP_COL = "CNTR_CODE", "LAU_ID", "LAU_NAME", "POP_2021"
AREA_COL, GEOM_AREA_COL, GEOM_VALID_COL = "AREA_KM2", "area_geom_km2", "geom_valid"
COORD_COLS = ["x_3035", "y_3035"]
MERGE_COUNTRY = "SK"               # country whose city parts are merged
SPLIT_MARKER = " - mestská časť"   # name pattern of city parts in the SK LAU names
SUM_COLS = [POP_COL, AREA_COL, GEOM_AREA_COL]
EXCLUDE_NONPOSITIVE_POP = True     # drop units with zero or missing population
MERGED_COL, N_PARTS_COL = "merged_lau_ids", "n_merged_parts"

out_dir = PROJECT_ROOT / OUT_DIR
units_path = out_dir / UNITS_FILE
units_rel = (Path(OUT_DIR) / UNITS_FILE).as_posix()

if units_path.is_file():
    units = pd.read_parquet(units_path)
    print("Loaded from file:", units_rel)
else:
    props = pd.read_parquet(out_dir / PROPS_FILE)
    cent = pd.read_parquet(out_dir / CENT_FILE)
    for df in (props, cent):
        df[ID_COL] = df[ID_COL].astype(str)
    for col in SUM_COLS:           # numeric inputs for the sums
        if col in cent.columns:
            cent[col] = pd.to_numeric(cent[col], errors="coerce")
    props[POP_COL] = pd.to_numeric(props[POP_COL], errors="coerce")

    full = cent.merge(props[[COUNTRY_COL, ID_COL, NAME_COL]],
                      on=[COUNTRY_COL, ID_COL], how="left", validate="one_to_one")
    is_part = (full[COUNTRY_COL] == MERGE_COUNTRY) & full[NAME_COL].fillna("").str.contains(SPLIT_MARKER, regex=False)
    full["city"] = full[NAME_COL].fillna("").str.split(SPLIT_MARKER, regex=False).str[0]

    # Merge the parts of each city into one unit
    merged_rows = []
    for city, g in full[is_part].groupby("city"):
        w = g[POP_COL]
        row = {COUNTRY_COL: MERGE_COUNTRY, ID_COL: f"{MERGE_COUNTRY}_{city}", NAME_COL: city,
               GEOM_VALID_COL: bool(g[GEOM_VALID_COL].all()),
               N_PARTS_COL: len(g), MERGED_COL: ";".join(g[ID_COL].astype(str))}
        for c in SUM_COLS:
            row[c] = g[c].sum()
        for c in COORD_COLS:       # population-weighted mean of the part centroids
            row[c] = float(np.average(g[c], weights=w))
        merged_rows.append(row)

    rest = full[~is_part].copy()
    rest[N_PARTS_COL] = 1
    rest[MERGED_COL] = rest[ID_COL].astype(str)
    cols = [COUNTRY_COL, ID_COL, NAME_COL, *SUM_COLS, *COORD_COLS, GEOM_VALID_COL, N_PARTS_COL, MERGED_COL]
    units = pd.concat([rest[cols], pd.DataFrame(merged_rows)[cols]], ignore_index=True)

    # Conservation check: merging must not change the national population totals
    before = props.groupby(COUNTRY_COL)[POP_COL].sum()
    after = units.groupby(COUNTRY_COL)[POP_COL].sum()
    assert np.allclose(before, after), "merging changed the national population totals"
    print("Population totals kept:", after.astype(int).to_dict())

    if EXCLUDE_NONPOSITIVE_POP:
        dropped = units[~(units[POP_COL] > 0)]
        print("Dropped units (non-positive population):")
        print(dropped[[COUNTRY_COL, ID_COL, NAME_COL, POP_COL]].to_string(index=False))
        units = units[units[POP_COL] > 0]

    units = units.sort_values([COUNTRY_COL, ID_COL]).reset_index(drop=True)
    units.to_parquet(units_path, index=False)
    print("Saved:", units_rel)

print("Units per country:", units[COUNTRY_COL].value_counts().to_dict())
print("Merged city units:", int((units[N_PARTS_COL] > 1).sum()),
      "| parts in them:", int(units.loc[units[N_PARTS_COL] > 1, N_PARTS_COL].sum()))

In [ ]:
import pandas as pd
from pathlib import Path

# Config: change only here
PROJECT_ROOT = Path("..")          # notebooks run from the notebooks folder
NEW_FILE = "data/interim/10_v4_inputs/hu_sk_lau_units.parquet"
OLD_FILE = "data/interim/hu_sk_lau_units.parquet"
COUNTRY_COL, ID_COL, NAME_COL = "CNTR_CODE", "LAU_ID", "LAU_NAME"
AREA_COLS = ["AREA_KM2", "area_geom_km2"]
MERGED_COL = "merged_lau_ids"
TOLERANCE = 1e-6                   # numeric differences below this count as equal

new = pd.read_parquet(PROJECT_ROOT / NEW_FILE)
if not (PROJECT_ROOT / OLD_FILE).is_file():
    print("Old table not found:", OLD_FILE)
else:
    old = pd.read_parquet(PROJECT_ROOT / OLD_FILE)
    key = [COUNTRY_COL, ID_COL]
    for df in (new, old):
        df[ID_COL] = df[ID_COL].astype(str)
    print("Rows: new", len(new), "| old", len(old))
    print("Columns only in new:", sorted(set(new.columns) - set(old.columns)),
          "| only in old:", sorted(set(old.columns) - set(new.columns)))

    both = new.merge(old, on=key, how="outer", suffixes=("_new", "_old"), indicator=True)
    print("Units only in new:", int((both["_merge"] == "left_only").sum()),
          "| only in old:", int((both["_merge"] == "right_only").sum()))
    same = both[both["_merge"] == "both"]

    common = [c for c in new.columns if c in old.columns and c not in key]
    bool_cols = [c for c in common if new[c].dtype == bool and old[c].dtype == bool]
    num_cols = [c for c in common if c not in bool_cols
                and pd.api.types.is_numeric_dtype(new[c]) and pd.api.types.is_numeric_dtype(old[c])]

    # Numeric columns: largest absolute difference
    for c in num_cols:
        diff = (same[f"{c}_new"] - same[f"{c}_old"]).abs().max()
        print(f"{c}: max abs difference {diff:.3g}" + ("" if diff <= TOLERANCE else "   <-- differs"))

    # Boolean and text columns: number of differing values
    for c in bool_cols + [NAME_COL, MERGED_COL]:
        if f"{c}_new" in same and f"{c}_old" in same:
            n_diff = int((same[f"{c}_new"].astype(str) != same[f"{c}_old"].astype(str)).sum())
            print(f"{c}: differing values {n_diff}")

    # Mean unit area per country and together, from the new table
    for col in AREA_COLS:
        per_country = new.groupby(COUNTRY_COL)[col].mean().round(2).to_dict()
        print(f"Mean {col}:", per_country, "| together:", round(float(new[col].mean()), 2))

In [ ]:
import requests
from pathlib import Path

# Config: change only here
PROJECT_ROOT = Path("..")          # notebooks run from the notebooks folder
SK_DIR = "data/raw/commuting/sk"
BASE_URL = "https://www.scitanie.sk/storage/app/media/dokumenty/"
FILES = ["SLO2021IV.csv",          # IV: obec / part of obec / ZSJ level
         "SLO2021III.csv"]         # III: country / kraj / okres level
PAGE_URL = "https://www.scitanie.sk/statisticky-lexikon-obci"
TIMEOUT_S = 60
MIN_BYTES = 1000                   # smaller files are treated as broken

out_dir = PROJECT_ROOT / SK_DIR
out_dir.mkdir(parents=True, exist_ok=True)

for name in FILES:
    target = out_dir / name
    rel = (Path(SK_DIR) / name).as_posix()
    if target.exists() and target.stat().st_size > MIN_BYTES:
        print("Already exists, skipped:", rel)
        continue
    try:
        resp = requests.get(BASE_URL + name, timeout=TIMEOUT_S)
        resp.raise_for_status()
        part = target.with_suffix(target.suffix + ".part")
        part.write_bytes(resp.content)
        part.replace(target)       # rename only after a complete download
        print(f"Downloaded: {rel} ({target.stat().st_size / 1e6:.2f} MB)")
    except Exception as exc:       # print the class name only, no paths
        print("Download failed:", name, "|", type(exc).__name__)
        print("Manual fix: download the file from", PAGE_URL, "and save it as", rel)

In [ ]:
import pandas as pd
from pathlib import Path

# Config: change only here
PROJECT_ROOT = Path("..")          # notebooks run from the notebooks folder
HU_DIR = "data/raw/commuting/hu"
UNITS_FILE = "data/interim/10_v4_inputs/hu_sk_lau_units.parquet"
UNITS_COUNTRY_COL, UNITS_ID_COL, UNITS_COUNTRY = "CNTR_CODE", "LAU_ID", "HU"
OUT_DIR = "data/interim/10_v4_inputs"
OUT_WIDE = "hu_commuting_wide.csv"
SOURCE_URL = "https://nepszamlalas2022.ksh.hu/adatbazis/"
CODE_COL, INDICATOR_COL, VALUE_COL, STATUS_COL = "TERUL_GEO5_FOGL", "FOGL_JEL", "OBS_VALUE", "OBS_STATUS"
INDICATORS = ["0_ING_T", "1_ING_T", "2_ING_T", "3_ING_T", "4_ING_T"]
IDENTITIES = [("0_ING_T", "2_ING_T", "3_ING_T"),   # first = second + third
              ("1_ING_T", "2_ING_T", "4_ING_T")]

hu_dir = PROJECT_ROOT / HU_DIR
hu_dir.mkdir(parents=True, exist_ok=True)
files = sorted(hu_dir.glob("*.csv"))

if len(files) != 1:
    print("Expected exactly one CSV in", HU_DIR, "| found:", [f.name for f in files])
    print("Manual fix: open", SOURCE_URL, "-> table WBS026 (employed population by settlement),")
    print("  select the settlement level only (3155 elements), export as CSV and save it in", HU_DIR)
else:
    raw = pd.read_csv(files[0], sep=";", dtype={CODE_COL: str})
    raw[VALUE_COL] = pd.to_numeric(raw[VALUE_COL], errors="coerce")
    print("Rows:", len(raw), "| status values:", raw[STATUS_COL].value_counts(dropna=False).to_dict())

    # One row per settlement, one column per indicator (the 'ING' group header is dropped)
    wide = (raw[raw[INDICATOR_COL].isin(INDICATORS)]
            .pivot(index=CODE_COL, columns=INDICATOR_COL, values=VALUE_COL)[INDICATORS])
    print("Settlements:", len(wide))
    print("Sum per indicator:", wide.sum().astype(int).to_dict())
    print("Missing per indicator:", wide.isna().sum().to_dict())

    # Identity checks on settlements where all values are present
    full = wide.dropna()
    print("Complete settlements:", len(full))
    for a, b, c in IDENTITIES:
        print(f"{a} = {b} + {c} holds for", int((full[a] == full[b] + full[c]).sum()), "of", len(full))

    # Key check against the units table
    units = pd.read_parquet(PROJECT_ROOT / UNITS_FILE)
    unit_codes = set(units.loc[units[UNITS_COUNTRY_COL] == UNITS_COUNTRY, UNITS_ID_COL].astype(str).str.strip())
    ksh_codes = set(wide.index.astype(str).str.strip())
    print("Codes: matched", len(ksh_codes & unit_codes),
          "| only KSH", len(ksh_codes - unit_codes), "| only units", len(unit_codes - ksh_codes))

    # Save the small wide table for analysis without re-running the notebook
    out_dir = PROJECT_ROOT / OUT_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    wide.to_csv(out_dir / OUT_WIDE)
    print("Saved:", (Path(OUT_DIR) / OUT_WIDE).as_posix())

In [ ]:
import pandas as pd
from pathlib import Path

# Config: change only here
PROJECT_ROOT = Path("..")          # notebooks run from the notebooks folder
SK_FILE = "data/raw/commuting/sk/SLO2021IV.csv"
SEP, ENCODING = ";", "cp1250"
N_PREVIEW = 2
MAX_UNIQUE = 8                     # columns with at most this many distinct values are listed

try:
    sk = pd.read_csv(PROJECT_ROOT / SK_FILE, sep=SEP, encoding=ENCODING, dtype=str)
except Exception as exc:           # print the class name only, no paths
    sk = None
    print("Read failed:", type(exc).__name__)

if sk is not None:
    print("Rows:", len(sk), "| columns:", sk.shape[1])
    for i, col in enumerate(sk.columns):
        print(i, repr(col))
    print()
    for col in sk.columns:
        values = sk[col].dropna().unique()
        if len(values) <= MAX_UNIQUE:
            print("Few values:", repr(col), "->", sorted(values.tolist()))
    print()
    print(sk.head(N_PREVIEW).T.to_string())

In [ ]:
import pandas as pd
from pathlib import Path

# Config: change only here
PROJECT_ROOT = Path("..")          # notebooks run from the notebooks folder
SK_FILE = "data/raw/commuting/sk/SLO2021IV.csv"
SEP, ENCODING = ";", "cp1250"
UNITS_FILE = "data/interim/10_v4_inputs/hu_sk_lau_units.parquet"
OUT_DIR, OUT_FILE = "data/interim/10_v4_inputs", "sk_zuj_outflow.csv"
LEVEL_COL, LEVEL_ZUJ = "Typ územnej úrovne", "obec"
OUT_COL = "Obyvateľstvo (SP) odchádzajúce za prácou"
CODE_COL = "Kód ZÚJ \n(obec/mestská časť/vojenský obvod)"
NAME_COL = "Názov ZÚJ \n(obec/mestská časť/vojenský obvod)"
KRAJ_COL, OKRES_COL = "Názov kraja", "Názov okresu"
POP_SP_COL = "Počet obyvateľov so súčasným pobytom (SP) spolu"
UNITS_COUNTRY_COL, UNITS_ID_COL, UNITS_COUNTRY = "CNTR_CODE", "LAU_ID", "SK"
MERGED_COL = "merged_lau_ids"      # ';'-joined part ids of the merged city units
N_PARTS_COL = "n_merged_parts"

sk = pd.read_csv(PROJECT_ROOT / SK_FILE, sep=SEP, encoding=ENCODING, dtype=str)
print("Rows per level:", sk[LEVEL_COL].value_counts(dropna=False).to_dict())

# Keep only the municipality (ZUJ) rows; the label row (level 'h') is dropped here
zuj = sk[sk[LEVEL_COL] == LEVEL_ZUJ][[CODE_COL, NAME_COL, KRAJ_COL, OKRES_COL, POP_SP_COL, OUT_COL]].copy()
zuj.columns = ["zuj_code", "zuj_name", "kraj", "okres", "pop_sp", "outflow"]

# Numbers use a space as thousands separator (also non-breaking spaces)
for col in ("pop_sp", "outflow"):
    cleaned = zuj[col].str.replace(r"\s+", "", regex=True)
    zuj[col] = pd.to_numeric(cleaned, errors="coerce")
    print(f"{col}: missing/non-numeric {int(zuj[col].isna().sum())} of {len(zuj)}")
zuj["zuj_code"] = zuj["zuj_code"].str.strip()
print("ZUJ rows:", len(zuj), "| unique codes:", zuj["zuj_code"].nunique())
print("Sum of outflow over ZUJ rows:", int(zuj["outflow"].sum()))

# Match with the SK units (single units and the parts of the merged cities)
units = pd.read_parquet(PROJECT_ROOT / UNITS_FILE)
sk_units = units[units[UNITS_COUNTRY_COL] == UNITS_COUNTRY]
single = set(sk_units.loc[sk_units[N_PARTS_COL] == 1, UNITS_ID_COL].astype(str).str.strip())
merged_parts = {p.strip() for s in sk_units.loc[sk_units[N_PARTS_COL] > 1, MERGED_COL].astype(str) for p in s.split(";")}
codes = set(zuj["zuj_code"])
print("Units SK:", len(sk_units), "| merged city units:", int((sk_units[N_PARTS_COL] > 1).sum()),
      "| parts in merged cities:", len(merged_parts))
print("ZUJ codes matched to single units:", len(codes & single), "| to merged parts:", len(codes & merged_parts),
      "| ZUJ codes in neither:", len(codes - single - merged_parts), "| single units without ZUJ:", len(single - codes))

# Upper bound of the double counting: the whole outflow of the ZUJ rows that were merged into cities
zuj["in_merged_city"] = zuj["zuj_code"].isin(merged_parts)
bound = int(zuj.loc[zuj["in_merged_city"], "outflow"].sum())
print("Outflow of ZUJ rows inside merged cities:", bound,
      "| share of the sum:", round(bound / zuj["outflow"].sum(), 4))

out_dir = PROJECT_ROOT / OUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)
zuj.to_csv(out_dir / OUT_FILE, index=False)
print("Saved:", (Path(OUT_DIR) / OUT_FILE).as_posix())

In [ ]:
import pandas as pd
from pathlib import Path

# Config: change only here
PROJECT_ROOT = Path("..")          # notebooks run from the notebooks folder
IN_DIR = "data/interim/10_v4_inputs"
HU_FILE, SK_FILE = "hu_commuting_wide.csv", "sk_zuj_outflow.csv"
HU_CODE_COL, HU_NC_COL = "TERUL_GEO5_FOGL", "4_ING_T"
SK_POP_COL, SK_OUT_COL, SK_MERGED_COL = "pop_sp", "outflow", "in_merged_city"
OUT_DIR, OUT_FILE = "data/processed/10_v4_inputs", "nc_summary.csv"

hu = pd.read_csv(PROJECT_ROOT / IN_DIR / HU_FILE, dtype={HU_CODE_COL: str})
sk = pd.read_csv(PROJECT_ROOT / IN_DIR / SK_FILE, dtype={"zuj_code": str})
sk = sk[sk[SK_POP_COL] > 0]        # units with non-positive population are not in the units table

rows = [
    {"country": "HU", "definition": f"sum of {HU_NC_COL} (daily incoming commuters, domestic)",
     "n_units": len(hu), "n_missing_values": int(hu[HU_NC_COL].isna().sum()),
     "nc_low": hu[HU_NC_COL].sum(), "nc_high": hu[HU_NC_COL].sum()},
    {"country": "SK", "definition": "sum of outflow over ZUJ rows; low = without ZUJ rows inside merged cities",
     "n_units": len(sk), "n_missing_values": int(sk[SK_OUT_COL].isna().sum()),
     "nc_low": sk.loc[~sk[SK_MERGED_COL], SK_OUT_COL].sum(), "nc_high": sk[SK_OUT_COL].sum()},
]
summary = pd.DataFrame(rows)
summary[["nc_low", "nc_high"]] = summary[["nc_low", "nc_high"]].astype(int)
summary["nc_range_share"] = (summary["nc_high"] - summary["nc_low"]) / summary["nc_high"]
print(summary.to_string(index=False))

out_dir = PROJECT_ROOT / OUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(out_dir / OUT_FILE, index=False)
print("Saved:", (Path(OUT_DIR) / OUT_FILE).as_posix())